# Module 05 — Shuffle & Sort

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import begin_module
from shared import config
from shared.utils import drop_iceberg_table

spark = begin_module('module-05-shuffle-sort')
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())

## Join Strategies

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50MB')
trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID').explain('formatted')

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID').explain('formatted')

## Storage Partitioned Join (SPJ)

In [ ]:
# Spark 4.x: use spark.conf.set (multi-statement SET in one sql() is not supported)
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')
spark.conf.set('spark.sql.sources.v2.bucketing.pushPartValues.enabled', 'true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping', 'true')
spark.conf.set('spark.sql.requireAllClusterKeysForCoPartition', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

In [ ]:
trips_a = trips.select('VendorID', 'PULocationID', 'total_amount', 'tpep_pickup_datetime')
trips_b = trips.select('VendorID', 'PULocationID', 'total_amount', 'tpep_dropoff_datetime')

drop_iceberg_table(spark, 'local.default.trips_a')
drop_iceberg_table(spark, 'local.default.trips_b')

trips_a.writeTo('local.default.trips_a').using('iceberg').partitionedBy('PULocationID').create()
trips_b.writeTo('local.default.trips_b').using('iceberg').partitionedBy('PULocationID').create()

In [ ]:
spj = spark.sql('''SELECT a.VendorID, a.total_amount, b.total_amount
    FROM local.default.trips_a a JOIN local.default.trips_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
spj.explain('formatted')
# SPJ success signal: NO Exchange / Shuffle before the join.
# You may see "KeyGroupedPartitioning" or "StoragePartitionJoin" — but absence of Exchange is enough.
spj.limit(10).show()

### Compare: SPJ off → Exchange appears

Disable SPJ and re-run the same join. The plan should now include **Exchange** (shuffle) before the join.

In [ ]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'false')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping', 'false')

no_spj = spark.sql('''SELECT a.VendorID, a.total_amount, b.total_amount
    FROM local.default.trips_a a JOIN local.default.trips_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
no_spj.explain('formatted')  # look for Exchange here

# Re-enable for the rest of the module
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping', 'true')

## Table Ordering

In [ ]:
drop_iceberg_table(spark, 'local.default.trips_ordered')

spark.table('local.default.trips_a').writeTo('local.default.trips_ordered').using('iceberg').create()
spark.sql('ALTER TABLE local.default.trips_ordered WRITE ORDERED BY (PULocationID, total_amount)')
spark.sql("SHOW TBLPROPERTIES local.default.trips_ordered").filter("key LIKE 'write.%' OR key = 'sort-order'").show(truncate=False)

## Key Takeaways

1. Broadcast avoids shuffle for small tables.
2. SPJ eliminates shuffle when Iceberg partitions align.
3. Table sort order improves reads but adds write cost.